In [1]:
import pandas as pd
import numpy as np

In [2]:
def load_netflix_file(filepath):

    data = []
    movie_id = None

    with open(filepath, "r") as f:

        for line in f:

            line = line.strip()

            if line.endswith(":"):
                movie_id = int(line[:-1])

            else:

                user_id, rating, date = line.split(",")

                data.append([
                    int(user_id),
                    movie_id,
                    int(rating),
                    date
                ])

    return pd.DataFrame(
        data,
        columns=[
            "user_id",
            "movie_id",
            "rating",
            "date"
        ]
    )

In [3]:
ratings = load_netflix_file(
    "../combined_data_1.txt"
)

In [4]:
sample_ratings = ratings.sample(
    n=1_000_000,
    random_state=42
)

sample_ratings.shape

(1000000, 4)

In [5]:
user_counts = sample_ratings.groupby(
    "user_id"
).size()

active_users = user_counts[
    user_counts >= 20
].index

ratings_filtered = sample_ratings[
    sample_ratings["user_id"].isin(active_users)
].copy()

ratings_filtered.shape

(50954, 4)

In [6]:
ratings_filtered["date"] = pd.to_datetime(
    ratings_filtered["date"]
)

ratings_filtered = ratings_filtered.sort_values(
    ["user_id", "date"]
)

In [7]:
train_list = []
test_list = []

for user_id, group in ratings_filtered.groupby(
    "user_id"
):

    train_list.append(
        group.iloc[:-5]
    )

    test_list.append(
        group.iloc[-5:]
    )

In [8]:
train_df = pd.concat(train_list)

test_df = pd.concat(test_list)

print(train_df.shape)
print(test_df.shape)

(41324, 4)
(9630, 4)


In [9]:
from surprise import Dataset
from surprise import Reader
from surprise import SVD
from surprise import accuracy

In [10]:
reader = Reader(
    rating_scale=(1,5)
)

train_data = Dataset.load_from_df(
    train_df[
        ["user_id","movie_id","rating"]
    ],
    reader
)

trainset = train_data.build_full_trainset()

In [11]:
testset = list(
    zip(
        test_df["user_id"],
        test_df["movie_id"],
        test_df["rating"]
    )
)

In [12]:
svd = SVD(
    n_factors=100,
    n_epochs=20,
    random_state=42
)

svd.fit(trainset)

In [13]:
predictions_svd = svd.test(
    testset
)

In [14]:
svd_rmse = accuracy.rmse(
    predictions_svd
)

svd_rmse

RMSE: 0.9809


np.float64(0.9808738908815409)

In [15]:
import pandas as pd
import numpy as np

eval_df = pd.DataFrame([
    (uid, iid, true_r, est)
    for uid, iid, true_r, est, _ in predictions_svd
], columns=[
    "user_id",
    "movie_id",
    "true_rating",
    "pred_rating"
])

eval_df.head()

,user_id,movie_id,true_rating,pred_rating
0,1333,1568,3,2.784338
1,1333,872,4,3.063209
2,1333,3559,3,2.584031
3,1333,884,3,2.628525
4,1333,1476,3,2.819936


In [16]:
eval_df = eval_df.sort_values(
    ["user_id", "pred_rating"],
    ascending=[True, False]
)

In [17]:
def map_at_k(df, k=10, threshold=3.5):

    APs = []

    for _, group in df.groupby("user_id"):

        top_k = group.head(k)

        relevant_count = 0
        precision_sum = 0

        for rank, (_, row) in enumerate(
            top_k.iterrows(),
            start=1
        ):

            if row["true_rating"] >= threshold:

                relevant_count += 1

                precision_sum += (
                    relevant_count / rank
                )

        if relevant_count > 0:

            APs.append(
                precision_sum / relevant_count
            )

    return np.mean(APs)

In [18]:
map10 = map_at_k(
    eval_df,
    k=10
)

print("MAP@10 =", round(map10, 4))

MAP@10 = 0.7615


In [19]:
eval_df.groupby("user_id").size().describe()

count    1926.0
mean        5.0
std         0.0
min         5.0
25%         5.0
50%         5.0
75%         5.0
max         5.0
dtype: float64